In [1]:
import time
import csv
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from datetime import datetime


In [ ]:
class TabelogScraper:
      def __init__(self, headless=False):
          """
          初期化
          :param headless: ヘッドレスモードで実行するか
          """
          options = webdriver.ChromeOptions()
          if headless:
              options.add_argument('--headless')
          options.add_argument('--no-sandbox')
          options.add_argument('--disable-dev-shm-usage')
          options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

          # webdriver-manager を使用して ChromeDriver を自動管理
          service = Service(ChromeDriverManager().install())
          self.driver = webdriver.Chrome(service=service, options=options)
          self.wait = WebDriverWait(self.driver, 10)

      def extract_store_info(self, url):
          """
          店舗情報を抽出
          :param url: 店舗ページのURL
          :return: 店舗情報の辞書
          """
          print(f"アクセス中: {url}")
          self.driver.get(url)
          time.sleep(1)  # ページ読み込み待機（短縮）

          store_data = {
              'store_id': '',
              'store_name': '',
              'genre': '',
              'rating': '',
              'url': url,
              'reviews': []
          }

          try:
              # 店舗ID（URLから抽出）
              store_data['store_id'] = url.rstrip('/').split('/')[-1]

              # 店舗名
              try:
                  store_name = self.driver.find_element(By.CSS_SELECTOR, '.display-name').text
                  store_data['store_name'] = store_name
              except NoSuchElementException:
                  print("店舗名が見つかりません")

              # ジャンル
              try:
                  genre_elem = self.driver.find_element(By.CSS_SELECTOR, '.rdheader-subinfo__item--cuisine')
                  store_data['genre'] = genre_elem.text
              except NoSuchElementException:
                  try:
                      # 別のセレクタを試す
                      genre_elem = self.driver.find_element(By.XPATH, "//th[contains(text(), 'ジャンル')]/following-sibling::td")
                      store_data['genre'] = genre_elem.text
                  except:
                      print("ジャンルが見つかりません")

              # 総合スコア（星評価）
              try:
                  rating_elem = self.driver.find_element(By.CSS_SELECTOR, '.rdheader-rating__score-val-dtl')
                  store_data['rating'] = rating_elem.text
              except NoSuchElementException:
                  try:
                      # 別のセレクタを試す
                      rating_elem = self.driver.find_element(By.CLASS_NAME, 'c-rating__val')
                      store_data['rating'] = rating_elem.text
                  except:
                      print("評価スコアが見つかりません")

              print(f"店舗情報取得完了: {store_data['store_name']} (評価: {store_data['rating']})")

          except Exception as e:
              print(f"店舗情報の取得エラー: {e}")

          return store_data

      def extract_reviews(self, store_url, max_pages=None):
          """
          レビューを抽出（全ページ）
          :param store_url: 店舗URL
          :param max_pages: 取得する最大ページ数（Noneの場合は全ページ）
          :return: レビューのリスト
          """
          reviews = []

          # レビューページに移動
          review_url = store_url.rstrip('/') + '/dtlrvwlst/'
          self.driver.get(review_url)
          time.sleep(1)  # ページ読み込み待機（短縮）

          page_count = 0

          while True:
              page_count += 1
              print(f"レビューページ {page_count} を取得中...")

              try:
                  # 全ての「もっと見る」ボタンをクリック
                  click_count = 0
                  while True:
                      try:
                          # 正しいセレクタで「もっと見る」ボタンを探す
                          button = self.driver.find_element(By.XPATH, "//span[@class='rvw-showall-trigger__target']")

                          if button and button.is_displayed():
                              # 親要素（クリック可能な要素）を取得
                              parent = button.find_element(By.XPATH, "..")
                              # スクロールしてボタンを表示
                              self.driver.execute_script("arguments[0].scrollIntoView(true);", parent)
                              time.sleep(0.1)  # スクロール待機（短縮）
                              # クリック
                              self.driver.execute_script("arguments[0].click();", parent)
                              click_count += 1
                              time.sleep(0.2)  # クリック待機（短縮）
                          else:
                              break
                      except NoSuchElementException:
                          # もう「もっと見る」ボタンがない
                          break
                      except Exception as e:
                          # その他のエラーで停止
                          break

                  if click_count > 0:
                      print(f"  {click_count} 個の「もっと見る」ボタンをクリックしました")

                  # レビューアイテムを取得
                  review_items = self.driver.find_elements(By.CSS_SELECTOR, '.rvw-item')

                  for item in review_items:
                      review = {}

                      try:
                          # レビュアー名
                          reviewer = item.find_element(By.CSS_SELECTOR, '.rvw-item__rvwr-name a').text
                          review['reviewer'] = reviewer
                      except:
                          review['reviewer'] = ''

                      try:
                          # レビュー評価
                          rating = item.find_element(By.CSS_SELECTOR, '.rvw-item__ratings .c-rating__val').text
                          review['review_rating'] = rating
                      except:
                          review['review_rating'] = ''

                      try:
                          # レビュー日付
                          date = item.find_element(By.CSS_SELECTOR, '.rvw-item__date').text
                          review['review_date'] = date
                      except:
                          review['review_date'] = ''

                      try:
                          # レビュー本文を取得
                          text_elem = item.find_element(By.CSS_SELECTOR, '.rvw-item__rvw-comment p')
                          review['review_text'] = text_elem.text
                      except:
                          review['review_text'] = ''

                      reviews.append(review)

                  print(f"  {len(review_items)} 件のレビューを取得")

                  # 最大ページ数チェック
                  if max_pages and page_count >= max_pages:
                      print(f"最大ページ数 {max_pages} に到達しました")
                      break

                  # 次のページボタンを探す
                  try:
                      next_button = self.driver.find_element(By.CSS_SELECTOR, '.c-pagination__arrow--next')

                      # リンクが無効かチェック
                      if 'c-pagination__arrow--disabled' in next_button.get_attribute('class'):
                          print("最終ページに到達しました")
                          break

                      # 次のページに移動
                      next_button.click()
                      time.sleep(1)  # ページ読み込み待機（短縮）

                  except NoSuchElementException:
                      print("次のページボタンが見つかりません（最終ページ）")
                      break

              except Exception as e:
                  print(f"レビュー取得エラー: {e}")
                  break

          print(f"合計 {len(reviews)} 件のレビューを取得しました")
          return reviews

      def scrape_store(self, url, max_review_pages=None):
          """
          1つの店舗の全データを取得
          :param url: 店舗URL
          :param max_review_pages: レビューの最大ページ数
          :return: 店舗データ
          """
          # 店舗情報を取得
          store_data = self.extract_store_info(url)

          # レビューを取得
          reviews = self.extract_reviews(url, max_review_pages)
          store_data['reviews'] = reviews

          return store_data

      def save_to_json(self, data, filename='tabelog_data.json'):
          """
          JSONファイルに保存
          """
          with open(filename, 'w', encoding='utf-8') as f:
              json.dump(data, f, ensure_ascii=False, indent=2)
          print(f"データを {filename} に保存しました")

      def save_to_csv(self, data, filename='tabelog_data.csv'):
          """
          CSVファイルに保存（フラット化）
          """
          rows = []
          for store in data:
              if store['reviews']:
                  for review in store['reviews']:
                      row = {
                          '店舗ID': store['store_id'],
                          '店舗名': store['store_name'],
                          'ジャンル': store['genre'],
                          '総合評価': store['rating'],
                          '店舗URL': store['url'],
                          'レビュアー': review.get('reviewer', ''),
                          'レビュー評価': review.get('review_rating', ''),
                          'レビュー日付': review.get('review_date', ''),
                          'レビュー本文': review.get('review_text', '')
                      }
                      rows.append(row)
              else:
                  # レビューがない場合
                  row = {
                      '店舗ID': store['store_id'],
                      '店舗名': store['store_name'],
                      'ジャンル': store['genre'],
                      '総合評価': store['rating'],
                      '店舗URL': store['url'],
                      'レビュアー': '',
                      'レビュー評価': '',
                      'レビュー日付': '',
                      'レビュー本文': ''
                  }
                  rows.append(row)

          if rows:
              with open(filename, 'w', encoding='utf-8', newline='') as f:
                  writer = csv.DictWriter(f, fieldnames=rows[0].keys())
                  writer.writeheader()
                  writer.writerows(rows)
              print(f"データを {filename} に保存しました")

      def close(self):
          """
          ブラウザを閉じる
          """
          self.driver.quit()


def main():
    """
    使用例
    """
    # スクレイパーを初期化（headless=True でヘッドレスモード）
    scraper = TabelogScraper(headless=False)

    # スクレイピングする店舗URLのリスト
    store_urls = [
        'https://tabelog.com/tokyo/A1310/A131002/13153985/',  # 例: サンプルURL
        # 他の店舗URLを追加
    ]

    all_data = []

    try:
        for url in store_urls:
            # 各店舗のデータを取得（max_review_pages で取得ページ数を制限可能）
            store_data = scraper.scrape_store(url, max_review_pages=5)  # 最初の5ページのみ
            all_data.append(store_data)

            # サーバーに負荷をかけないよう待機
            time.sleep(1)  # 待機時間短縮

        # データを保存
        scraper.save_to_json(all_data, 'tabelog_data.json')
        scraper.save_to_csv(all_data, 'tabelog_data.csv')

    finally:
        # ブラウザを閉じる
        scraper.close()


if __name__ == '__main__':
    main()

## 東京規模


In [ ]:
class TokyoTabelogScraper(TabelogScraper):
      """
      東京の全店舗をシンプルに取得
      """

      def get_all_tokyo_restaurants(self, max_pages=None):
          """
          東京の全店舗URLを取得
          :param max_pages: 取得する最大ページ数（Noneで全ページ）
          :return: 店舗URLのリスト
          """
          print("東京の全店舗URLを取得中...")
          restaurant_urls = []

          # 東京のリストページ
          base_url = 'https://tabelog.com/tokyo/'
          self.driver.get(base_url)
          time.sleep(1)  # 待機時間短縮

          page_count = 0

          while True:
              page_count += 1
              print(f"\nリストページ {page_count} を処理中...")

              try:
                  # 店舗リンクを全て取得
                  links = []

                  # セレクタ1: list-rst__rst-name-target
                  try:
                      links = self.driver.find_elements(By.CSS_SELECTOR, 'a.list-rst__rst-name-target')
                  except:
                      pass

                  # セレクタ2: 店舗名リンク
                  if not links:
                      try:
                          links = self.driver.find_elements(By.CSS_SELECTOR, 'a.list-rst__name')
                      except:
                          pass

                  # セレクタ3: href パターンマッチ
                  if not links:
                      all_links = self.driver.find_elements(By.TAG_NAME, 'a')
                      links = [l for l in all_links if l.get_attribute('href') and '/tokyo/A' in l.get_attribute('href')]

                  # URLを抽出
                  page_urls = []
                  for link in links:
                      url = link.get_attribute('href')
                      if url and '/tokyo/A' in url and url not in restaurant_urls:
                          # 店舗詳細ページのみ（/lst/やエリアページを除外）
                          if '/lst/' not in url and url.count('/') >= 6:
                              base_url_clean = url.split('?')[0].rstrip('/')
                              if base_url_clean not in restaurant_urls and base_url_clean not in page_urls:
                                  page_urls.append(base_url_clean)

                  restaurant_urls.extend(page_urls)
                  print(f"  このページから {len(page_urls)} 件取得（累計: {len(restaurant_urls)} 件）")

                  # 最大ページ数チェック
                  if max_pages and page_count >= max_pages:
                      print(f"\n最大ページ数 {max_pages} に到達しました")
                      break

                  # 次のページに移動
                  try:
                      next_button = self.driver.find_element(By.CSS_SELECTOR, '.c-pagination__arrow--next')

                      if 'c-pagination__arrow--disabled' in next_button.get_attribute('class'):
                          print("\n最終ページに到達しました")
                          break

                      next_button.click()
                      time.sleep(1)  # 待機時間短縮

                  except NoSuchElementException:
                      print("\n次のページボタンが見つかりません（最終ページ）")
                      break

              except Exception as e:
                  print(f"ページ処理エラー: {e}")
                  break

          print(f"\n合計 {len(restaurant_urls)} 件の店舗URLを取得しました")
          return restaurant_urls

      def scrape_tokyo_all(self, output_json='tabelog_tokyo_all.json', output_csv='tabelog_tokyo_all.csv', max_pages_list=None, max_review_pages=None):
          """
          東京の全店舗をスクレイピング（1つのファイルに統合）
          :param output_json: 出力JSONファイル名
          :param output_csv: 出力CSVファイル名
          :param max_pages_list: リストページの最大ページ数（Noneで全ページ）
          :param max_review_pages: 各店舗のレビュー最大ページ数（Noneで全ページ）
          :return: 全店舗データ
          """
          all_data = []

          # 1. 全店舗URLを取得
          print(f"{'='*70}")
          print("ステップ1: 東京の全店舗URLを取得")
          print(f"{'='*70}")

          restaurant_urls = self.get_all_tokyo_restaurants(max_pages=max_pages_list)

          if not restaurant_urls:
              print("店舗URLが取得できませんでした")
              return all_data

          # 2. 各店舗をスクレイピング
          print(f"\n{'='*70}")
          print(f"ステップ2: 各店舗のデータを取得")
          print(f"総店舗数: {len(restaurant_urls)} 件")
          print(f"出力先: {output_json}, {output_csv}")
          print(f"{'='*70}\n")

          for idx, url in enumerate(restaurant_urls, 1):
              print(f"\n{'='*70}")
              print(f"進捗: {idx}/{len(restaurant_urls)} ({idx/len(restaurant_urls)*100:.2f}%)")
              print(f"URL: {url}")
              print(f"{'='*70}")

              try:
                  store_data = self.scrape_store(url, max_review_pages=max_review_pages)
                  all_data.append(store_data)

                  # 10件ごとに進捗を同じファイルに上書き保存
                  if idx % 10 == 0:
                      self.save_to_json(all_data, output_json)
                      self.save_to_csv(all_data, output_csv)
                      print(f"\n*** 進捗保存: {len(all_data)} 件 ({idx}/{len(restaurant_urls)}) ***")
                      print(f"*** ファイル: {output_json}, {output_csv} ***\n")

                  # サーバーに負荷をかけないよう待機
                  time.sleep(1)  # 待機時間短縮

              except Exception as e:
                  print(f"\n!!! エラー: {e}")
                  print(f"!!! この店舗をスキップして続行します\n")
                  # エラーが起きても途中保存
                  if idx % 10 == 0:
                      self.save_to_json(all_data, output_json)
                      self.save_to_csv(all_data, output_csv)
                  continue

          # 最終保存
          self.save_to_json(all_data, output_json)
          self.save_to_csv(all_data, output_csv)

          print(f"\n{'='*70}")
          print(f"完了！")
          print(f"取得件数: {len(all_data)}/{len(restaurant_urls)} 件")
          print(f"保存先: {output_json}, {output_csv}")
          print(f"{'='*70}")

          return all_data


# 実行関数

def run_tokyo_full_scraping():
    """
    東京全店舗のスクレイピング実行（138,383件）
    全データを1つのJSONと1つのCSVに保存
    """
    print("="*70)
    print("東京全店舗スクレイピング開始")
    print("="*70)

    scraper = TokyoTabelogScraper(headless=True)  # ヘッドレスで高速化

    try:
        all_data = scraper.scrape_tokyo_all(
            output_json='tabelog_tokyo_all.json',
            output_csv='tabelog_tokyo_all.csv',
            max_pages_list=None,  # 全ページ
            max_review_pages=None  # 全レビュー
        )

        print(f"\n最終結果:")
        print(f"  取得件数: {len(all_data)} 件")
        print(f"  JSON: tabelog_tokyo_all.json")
        print(f"  CSV: tabelog_tokyo_all.csv")

    except KeyboardInterrupt:
        print("\n\n中断されました。現在までのデータは保存されています。")
    except Exception as e:
        print(f"\n\nエラーが発生しました: {e}")
        print("現在までのデータは保存されています。")
    finally:
        scraper.close()


# テスト用（少数の店舗で動作確認）
def test_tokyo_scraping():
    """
    テスト実行（最初の2ページのみ）
    """
    print("="*70)
    print("テストモード")
    print("="*70)

    scraper = TokyoTabelogScraper(headless=False)

    try:
        all_data = scraper.scrape_tokyo_all(
            output_json='tabelog_tokyo_test.json',
            output_csv='tabelog_tokyo_test.csv',
            max_pages_list=2,  # 最初の2ページのみ
            max_review_pages=2  # レビュー2ページまで
        )

        print(f"\nテスト完了:")
        print(f"  取得件数: {len(all_data)} 件")

    finally:
        scraper.close()

In [25]:
run_tokyo_full_scraping()

東京全店舗スクレイピング開始
ステップ1: 東京の全店舗URLを取得
東京の全店舗URLを取得中...
ステップ1: 東京の全店舗URLを取得
東京の全店舗URLを取得中...

リストページ 1 を処理中...
  このページから 20 件取得（累計: 20 件）

リストページ 1 を処理中...
  このページから 20 件取得（累計: 20 件）

リストページ 2 を処理中...

リストページ 2 を処理中...
  このページから 20 件取得（累計: 40 件）
  このページから 20 件取得（累計: 40 件）

リストページ 3 を処理中...
  このページから 20 件取得（累計: 60 件）

リストページ 3 を処理中...
  このページから 20 件取得（累計: 60 件）

リストページ 4 を処理中...

リストページ 4 を処理中...
  このページから 20 件取得（累計: 80 件）
  このページから 20 件取得（累計: 80 件）

リストページ 5 を処理中...
  このページから 20 件取得（累計: 100 件）

リストページ 5 を処理中...
  このページから 20 件取得（累計: 100 件）

リストページ 6 を処理中...
  このページから 20 件取得（累計: 120 件）

リストページ 6 を処理中...
  このページから 20 件取得（累計: 120 件）

リストページ 7 を処理中...
  このページから 20 件取得（累計: 140 件）

リストページ 7 を処理中...
  このページから 20 件取得（累計: 140 件）

リストページ 8 を処理中...
  このページから 20 件取得（累計: 160 件）

リストページ 8 を処理中...
  このページから 20 件取得（累計: 160 件）

リストページ 9 を処理中...
  このページから 20 件取得（累計: 180 件）

リストページ 9 を処理中...
  このページから 20 件取得（累計: 180 件）

リストページ 10 を処理中...
  このページから 20 件取得（累計: 200 件）

リストページ 10 を処理中...
  このページから 20 件取得（累計: 200 

In [6]:
class Tokyo23KuTabelogScraper(TabelogScraper):
    """
    東京23区の店舗データを柔軟に取得するスクレイパー
    """

    def get_urls_by_area(self, area_code, target_count):
        """
        特定のエリアコードから指定した店舗数分のURLを回収する
        """
        area_urls = []
        page = 1
        
        while len(area_urls) < target_count:
            # 食べログのエリア別リストURL（ページ番号付与）
            url = f"https://tabelog.com/tokyo/{area_code}/rstLst/{page}/"
            self.driver.get(url)
            time.sleep(1)  # 待機時間短縮
            
            # 店舗リンクを取得（既存のセレクタを使用）
            links = self.driver.find_elements(By.CSS_SELECTOR, 'a.list-rst__rst-name-target')
            if not links:
                links = self.driver.find_elements(By.CSS_SELECTOR, 'a.list-rst__name')
            
            if not links:
                print(f"    [!] ページ {page} で店舗が見つかりませんでした。終了します。")
                break
            
            for link in links:
                href = link.get_attribute('href').split('?')[0].rstrip('/')
                if '/tokyo/A' in href and href.count('/') >= 6 and '/lst/' not in href:
                    if href not in area_urls:
                        area_urls.append(href)
                
                if len(area_urls) >= target_count:
                    break
            
            print(f"    ページ {page}: 現在 {len(area_urls)}/{target_count} 件のURLを確保")
            
            # 次のページボタンがあるか確認
            try:
                next_btn = self.driver.find_element(By.CSS_SELECTOR, '.c-pagination__arrow--next')
                if 'c-pagination__arrow--disabled' in next_btn.get_attribute('class'):
                    break
                page += 1
            except:
                break
                
        return area_urls[:target_count]

    def execute_23ku_workflow(self, stores_per_area, output_json, output_csv, max_review_pages=None, is_debug=False):
        """
        23区を巡回してデータを取得する汎用メインワークフロー
        """
        all_data = []
        area_codes = [f"C131{i:02d}" for i in range(1, 24)]
        
        mode_name = "【デバッグモード】" if is_debug else "【本番スクレイピング】"
        print(f"\n{'='*70}\n{mode_name}\n各区目標: {stores_per_area} 店舗 / 口コミ: {'制限なし' if max_review_pages is None else max_review_pages}\n{'='*70}")

        for code in area_codes:
            print(f"\nエリアコード {code} を処理中...")
            
            # 1. URLを必要数取得
            target_urls = self.get_urls_by_area(code, stores_per_area)
            
            # 2. 各URLに対して詳細情報を取得
            for idx, url in enumerate(target_urls, 1):
                try:
                    print(f"  [{code}] 進捗: {idx}/{len(target_urls)}店舗目")
                    store_data = self.scrape_store(url, max_review_pages=max_review_pages)
                    all_data.append(store_data)
                    
                    # 5件ごとに保存（データ保護）
                    if len(all_data) % 5 == 0:
                        self.save_to_json(all_data, output_json)
                        self.save_to_csv(all_data, output_csv)
                    
                    time.sleep(1)  # 待機時間短縮
                except Exception as e:
                    print(f"  [!] エラー: {url} の取得をスキップします ({e})")
                    continue
        
        # 最終保存
        self.save_to_json(all_data, output_json)
        self.save_to_csv(all_data, output_csv)
        print(f"\n{'='*70}\n完了！合計 {len(all_data)} 店舗のデータを取得しました。\n{'='*70}")
        return all_data

# --- 実行用関数（クラス外） ---

def run_23ku_debug(stores_per_area=20):
    """
    各区指定数（デフォルト20店舗）ずつのデバッグ実行
    """
    scraper = Tokyo23KuTabelogScraper(headless=False) # 動作確認のため見える状態で実行
    try:
        scraper.execute_23ku_workflow(
            stores_per_area=stores_per_area,
            output_json='tabelog_debug_23ku.json',
            output_csv='tabelog_debug_23ku.csv',
            max_review_pages=None, # 全口コミ
            is_debug=True
        )
    finally:
        scraper.close()

def run_23ku_scraping(stores_per_area=1000):
    """
    本番用：各区指定数（デフォルト100店舗）ずつのスクレイピング
    """
    scraper = Tokyo23KuTabelogScraper(headless=True) # 本番は高速化のためヘッドレス
    try:
        scraper.execute_23ku_workflow(
            stores_per_area=stores_per_area,
            output_json='tabelog_production_23ku.json',
            output_csv='tabelog_production_23ku.csv',
            max_review_pages=None,
            is_debug=False
        )
    finally:
        scraper.close()

# # --- メイン実行 ---
# if __name__ == '__main__':
#     # デバッグしたい場合（各区20店舗）
#     run_23ku_debug(stores_per_area=20)
    
    # 本番動かしたい場合（各区100店舗なら以下のコメントを外す）
    # run_23ku_scraping(stores_per_area=100)

In [7]:
run_23ku_scraping(stores_per_area=1000)


【本番スクレイピング】
各区目標: 1000 店舗 / 口コミ: 制限なし

エリアコード C13101 を処理中...
    ページ 1: 現在 20/1000 件のURLを確保
    ページ 2: 現在 40/1000 件のURLを確保
    ページ 3: 現在 60/1000 件のURLを確保
    ページ 4: 現在 80/1000 件のURLを確保
    ページ 5: 現在 100/1000 件のURLを確保
    ページ 6: 現在 120/1000 件のURLを確保
    ページ 7: 現在 140/1000 件のURLを確保
    ページ 8: 現在 160/1000 件のURLを確保
    ページ 9: 現在 180/1000 件のURLを確保
    ページ 10: 現在 200/1000 件のURLを確保
    ページ 11: 現在 220/1000 件のURLを確保
    ページ 12: 現在 240/1000 件のURLを確保
    ページ 13: 現在 260/1000 件のURLを確保
    ページ 14: 現在 280/1000 件のURLを確保
    ページ 15: 現在 300/1000 件のURLを確保
    ページ 16: 現在 320/1000 件のURLを確保
    ページ 17: 現在 340/1000 件のURLを確保
    ページ 18: 現在 360/1000 件のURLを確保
    ページ 19: 現在 380/1000 件のURLを確保
    ページ 20: 現在 400/1000 件のURLを確保
    ページ 21: 現在 420/1000 件のURLを確保
    ページ 22: 現在 440/1000 件のURLを確保
    ページ 23: 現在 460/1000 件のURLを確保
    ページ 24: 現在 480/1000 件のURLを確保
    ページ 25: 現在 500/1000 件のURLを確保
    ページ 26: 現在 520/1000 件のURLを確保
    ページ 27: 現在 540/1000 件のURLを確保
    ページ 28: 現在 560/1000 件のURLを確保
    ページ 29: 現在 580/1000 件のU

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
0   chromedriver                        0x00000001045dcb2c cxxbridge1$str$ptr + 3033596
1   chromedriver                        0x00000001045d4b30 cxxbridge1$str$ptr + 3000832
2   chromedriver                        0x00000001040ce970 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 73784
3   chromedriver                        0x0000000104109538 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 314368
4   chromedriver                        0x000000010413145c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 477988
5   chromedriver                        0x0000000104130890 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 474968
6   chromedriver                        0x000000010409d6f8 chromedriver + 10